In [ ]:
import requests
import time
import pandas as pd
import csv
import re
import os
from datetime import datetime
from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer, util
from dotenv import load_dotenv
load_dotenv()
guardian_api_token = os.getenv('guardian_TOKEN') 
#nyt_api_token = os.getenv('nyt_TOKEN')


In [ ]:
def get_content_from_guardian(searches,start_date,end_date,api_key):
    content = []
    #is it 50 articels per page, so we go through mutliple pages that match with the same date needed
    for i in searches:
        page = 1
        while True:
            query = requests.get('https://content.guardianapis.com/search',params={'q' :i,'from-date':start_date,'to-date': end_date,'api-key': api_key,'page-size'  : 50,'page'       : page,
                                                                                    'show-fields': 'headline,body'}
                                                                                    )
            data = query.json()['response']
            content.extend(data['results'])
            if page >= data['pages']:
                break
            page += 1
    return content
#fields is a dict of title and content

searches = ['(Iran OR Israel) AND oil AND war',
    '"oil price" AND Iran',
    '"energy market" AND Iran',
    '"oil volatility" AND Iran',
    '"Strait of Hormuz"',
    'Iran AND sanctions AND oil',
    'energy volatility',
    'investor sentiment',
    '"market volatility" AND energy',]

raw_guardian_df = pd.DataFrame(get_content_from_guardian(searches, '2025-06-01',str(datetime.today().date()),guardian_api_token))

In [ ]:
guardian_df = raw_guardian_df.copy()

In [ ]:
guardian_df['date'] = pd.to_datetime(guardian_df['webPublicationDate']).dt.strftime('%Y%m%d')
guardian_df.drop_duplicates(subset=['webUrl'], inplace=True)
guardian_df = guardian_df[['date','fields']]
guardian_df.reset_index(drop=True, inplace=True)

In [ ]:
guardian_df['headline'] = None
guardian_df['body_text'] = None

for i, item in enumerate(guardian_df['fields']):
    guardian_df.at[i, 'headline'] = item['headline']
    soup = BeautifulSoup(item['body'])
    body_text = soup.get_text()
    guardian_df.at[i, 'body_text'] = body_text

In [ ]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
embeddings = model.encode('oil price energy markets Iran war investor sentiment volatility crude supply',convert_to_tensor=True)

def relevancy(headline):
    temp = model.encode(headline, convert_to_tensor=True)
    return util.cos_sim(temp,embeddings).item()

guardian_df['relevancy'] = guardian_df['headline'].apply(relevancy)
guardian_df = guardian_df[guardian_df['relevancy'] > 0.38]

    

In [ ]:
guardian_df = guardian_df[['date','headline','body_text']]
guardian_df.reset_index(drop=True, inplace=True)
guardian_df.to_csv('guardian_data_since_20250601.csv', index=False, encoding='utf-8')


In [ ]:
# def get_content_from_nyt(input,start_date,end_date,api_key):
#     content = []
#     page = 0
#     while True:
#         time.sleep(7)  #max 10 queries per min
#         query = requests.get('https://api.nytimes.com/svc/search/v2/articlesearch.json',params={'q' :input,'begin_date':start_date,'end_date': end_date,'api-key': api_key,'page': page})
#         data = query.json()['response']
#         content.extend(data['docs'])
#         if len(data['docs']) < 10:  # NYT returns 10 per page
#             break
#         page += 1
#     return (content)

# nyt_df = pd.DataFrame(get_content_from_nyt('Iran Israel oil energy', '20250601',(datetime.today().date()).strftime('%Y%m%d'),nyt_api_token))
# nyt_df = nyt_df[['pub_date','headline','abstract']]

In [ ]:
# nyt_df.to_csv('nyt_data_since_20250601.csv', index=False, encoding='utf-8')